# Manager lifecycle: compose → fit → predict → encrypt → submit

Condensed path for a **strategist** against a **confidential** Orion vault.

1. Compose a modular estimator.

2. Bridge with `IntentSession`:
    - `fit`
    - `predict`
    - `encrypt`
    - `submit`

In [1]:
from __future__ import annotations

import os
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
from eth_account import Account
from orion_finance_sdk_py import (
    IntentSession,
    OrionConfig,
    OrionEncryptedVault,
    PriceAdapterRegistry,
    ReturnSeries,
)
from orion_finance_sdk_py.stats import chronological_split, daily_rfr, rfr_decimal
from orion_finance_sdk_py.utils import checksum_address
from skfolio import RiskMeasure
from skfolio.optimization import MeanRisk, ObjectiveFunction

load_dotenv()

LOOKBACK_DAYS = int(os.getenv("ORION_LIFECYCLE_LOOKBACK_DAYS", "7"))
vault_address = checksum_address(os.environ["ORION_VAULT_ADDRESS"])
config = OrionConfig()
assert config.is_encrypted_vault(vault_address), "Expected an encrypted vault"

vault = OrionEncryptedVault(contract_address=vault_address)
strategist_key = os.environ["STRATEGIST_PRIVATE_KEY"]
strategist_addr = Account.from_key(strategist_key).address

print("Vault:", vault_address)
print("Manager:", vault.manager_address)
print("Strategist (onchain):", vault.strategist_address)
print("Strategist (env key):", strategist_addr)

Vault: 0x9a2E639Aa0359CD8ce27E3C74B23Ee5e3E5DdEf4
Manager: 0xF97e66d9602de128FdFC8bcBe6651f1FC89c8AF2
Strategist (onchain): 0xF97e66d9602de128FdFC8bcBe6651f1FC89c8AF2
Strategist (env key): 0xF97e66d9602de128FdFC8bcBe6651f1FC89c8AF2


## Universe returns

Whitelist prices → `ReturnSeries` → excess returns → chronological split.

In [2]:
registry = PriceAdapterRegistry()
end = datetime.now(timezone.utc)
start = end - timedelta(days=LOOKBACK_DAYS)
series = registry.price_history(start=start, end=end)

names = dict(zip(config.whitelisted_assets, config.whitelisted_asset_names))
address_by_label = {name: addr for addr, name in names.items()}

rs = ReturnSeries.from_price_history(
    series,
    decimals=registry.price_adapter_decimals,
    names=names,
)
rfr = rfr_decimal(config.risk_free_rate)
train, test = chronological_split(rs.returns, test_size=0.25)

## Pipeline

Compose skfolio → bind `IntentSession` → fit → predict → encrypt → submit.

In [3]:
# Compose an unfitted skfolio estimator (swap risk_measure / priors freely).
model = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.SEMI_VARIANCE,  # Sortino; try VARIANCE / CVAR / ...
    risk_free_rate=daily_rfr(rfr),
    portfolio_params={"annualized_factor": 365.0},
)

In [4]:
# Bind the estimator and label→address map to the vault lifecycle session.
session = IntentSession(model, address_by_label=address_by_label)

In [5]:
# Prepare returns and fit; weights are labeled fractions summing to 1.
session.fit(train)

/Users/matteoettam09/Developer/sdks/orion-finance-sdk-py/.venv/lib/python3.13/site-packages/skfolio/moments/covariance/_base.py:362: UserWarning: The covariance matrix is not positive definite. The Clipping algorithm will be used to find the nearest positive definite covariance.
  covariance = cov_nearest(


In [6]:
session.weights.sort_values(ascending=False).head()

Keyrock USDC          0.048237
EVK Vault ePYUSD-6    0.038570
Frax                  0.025229
EVK Vault eRLUSD-7    0.023028
EVK Vault eUSDC-80    0.021761
dtype: float64

In [7]:
# Seal the fitted intent with Orion HPKE (inspect / offline; does not broadcast).
ciphertext = session.encrypt()

In [12]:
ciphertext

b'~\x0c*\xff\xe5[\xc5\xf9\x10)\xd6\xc7P\x85\xc64\xdd\x89\xe9\x9e\xb2\x0f\x89\x0bS4\xf8`\xde\x8cF\n\x05\xd2t\xa6\xb0VgL\xd6\r\xf8Zy\xc8\xa0[\xbc\x00P\xf8\xe3\x93\xd4&G\x8d\xaf\x95\xd2\xc0`\x9d\xf9\xed[\xfe\x0fn]\xfb\xa9\xc8\xbbe\xa2n,\x19S\x869\xda\xb0-\xc8\xb7]\x86AS\t)\x92D.I\'\xd6\xac\x9d\x8d\x88\xee\xb6I\xf3\xf6\xb3\x93\x82\xc0C\xa6\xeb~O\xa5\xaf\x18\x14\x9dU\x9b\x8e\x05\x0fv\xffu\xfd\x1c\x8d\xbf\x92$/0\x02ng\t\xf1A\xe9La\x1f\x01E1\xdc4D\xb9\xc2\xec\x9eS\xbb\xd0\x9fYH\xdc\xdd\x89w?\xec\x01\x18\x8d\xb7\xa7\'\xae\x9a\x85n \xaf\x82\x7f\xf7\xdf\x9a>S\xa9jy\xeaspWD\xad\xaef~\xbf\x8aO\xd9\xc7\xedp\x0c\xe9\xd4\xdb\x1b\xf6J\xfc\xaf\xd0\x16\xbeQ\x8an\x12\xe6\xa4?`H\xdb\xcb\x1eH\xa0\x1ag\xba\x05\x80\x93\xf7\xad\x8b\x8b\xf3\xf9\xef\x12\x89\xae\xb0\xc2\xea\xac\x14\xef\rE\xa4\xb9\xc2\xcd\xa6B[\xd2\xd8\xd6\xbef\xf9\x17\x93\xa7"\x85\xd8\x05\xe2`\xc4\x00\x834+\x9e\x7fR\x89\x95\x07\xf5\x16\xc0\xc4\x13i%\xc2"a\xfe\xf8U\xb9\x07\xe1h\xea\x8b\xf3\x16\x96\x8f\xf2K\n\xee\x91HV\x11\x9f\x8a\xf2y`\xce\x05\xe

In [9]:
# Broadcast the intent.
tx = session.submit()
tx.tx_hash

'2557e3cc311602175ee0d35a455d3fe2ef9bd64ec7f87cff48df2f3e9523de32'